In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab-mistral/notebooks/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 01 · The arithmetic of agent scale — Mistral's API or your own GPUs

**What you'll learn.** How to get from "100,000 conversations a day" to tokens per minute, turns in flight,
a rate-limit request, a GPU fleet and dollars — in the two minutes a whiteboard gives you — and how to say at
what volume self-hosting pays for itself. The model is `scalelab/capacity.py` (the demand and the money) and
`scalelab/serving.py` (what one vLLM replica delivers); read them once, then use them.

> **In a design review.** The customer's architect asks "hosted or self-hosted?". The sentence to say out loud:
> *"The unit of work is a turn; a turn is ~2.2 model calls of ~5k tokens; so 100k conversations a day is about
> 14 million tokens a minute at peak. On the API that is a 60-RPS / 19 M-TPM limit I have to ask Mistral for;
> on your GPUs it is eleven H100s sized for the peak — and at this volume the two cost about the same on
> committed GPUs, so your residency requirement decides, and I can tell you what it costs."*

In [ ]:
import sys, asyncio, inspect, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 160)
CLOCK.reset(0.02)   # 50x faster than real time

## 1. The chain of multiplications

conversations/day → conversations/s → turns/s → model calls/s → tokens/min. Every later number hangs off this.

In [ ]:
from scalelab.capacity import Scenario, plan, plan_markdown, cost_per_call, hosted_costs, fleet, PRICES, FREE_TIER
from scalelab.serving import replica, OPEN_MODELS, GPUS, GPU_USD_PER_HOUR
s = Scenario()          # the anchor scenario: Meridian Mobile, a Singapore telco, 100k conversations/day
p = plan(s)
pd.DataFrame(p["rates"]).round(2)

In [ ]:
tokens = pd.DataFrame(p["tokens"]).T
tokens[["input_tpm", "uncached_input_tpm", "output_tpm", "total_tpm"]].div(1e6).round(2).rename(columns=lambda c: c + " (M)").assign(vs_limit=tokens.vs_limit.round(2))

In [ ]:
ax = tokens.total_tpm.div(1e6).plot.bar(figsize=(6, 3.5), rot=0, title="Total tokens per minute vs the API limit")
ax.axhline(s.tpm_limit / 1e6, ls="--", c="grey", label=f"assumed workspace limit ({s.tpm_limit/1e6:.0f} M TPM)")
ax.axhline(FREE_TIER["tpm"] / 1e6, ls=":", c="red", label="free tier (0.5 M TPM)"); ax.set_ylabel("M tokens / min"); ax.legend(); plt.tight_layout()

Mistral's paid-tier limits are shown only in the console (tiers unlock at $20 / $100 / $500 / $2,000 of cumulative
billing; above that you write to support with *target RPS, model, TPM and tokens per month*). So the capacity plan
**is** the rate-limit request. The incident (×10) is 2.4× even the assumed limit — nothing you ask for makes 48 M TPM
appear; the incident has to be *shaped* (notebooks 03/04). The peak can be bought (a higher limit, the Priority Tier)
or reduced (fewer tokens per call).

## 2. Little's law: concurrency is latency × rate

Things in the system = arrival rate × time each spends in it. Turns in flight size orchestrator memory and model
concurrency; concurrent *sessions* (users thinking between turns) size open connections and hot state.

In [ ]:
conc = pd.DataFrame({k: v for k, v in p["concurrency"].items() if isinstance(v, dict)}).round(0)
print(conc)
print(f"\nthe assumed {s.tpm_limit/1e6:.0f} M TPM limit sustains ≈ {p['concurrency']['max_turns_per_s_for_limit']:.1f} turns/s "
      f"≈ {p['concurrency']['max_inflight_for_limit']:.0f} turns in flight — the starting in-flight cap on the API")
print("the request to send Mistral support:", p["hosted"]["limit_request"])

## 3. Hosted: cost per conversation

Cached input is billed at 10 % (64-token blocks, `prompt_cache_key`). Mistral Small 4 is the default agent model;
Medium 3.5 (τ³-Telecom 91.4) is the escalation model at 10 % of calls; the Priority Tier is +75 %, a regional
(EU/US) endpoint +10 %. Route and cache *first*, because every later percentage applies to the new baseline.

In [ ]:
cost = pd.DataFrame({"$/conversation": p["hosted"]["cost_per_conversation"], "$/month": p["hosted"]["monthly_usd"]}).round(4)
cost

In [ ]:
ax = cost["$/conversation"].plot.barh(figsize=(7, 3.8), title="$ per conversation on Mistral's API"); ax.set_xlabel("USD"); plt.tight_layout()

## 4. Self-hosted: what one replica delivers

A vLLM replica has two budgets. **Memory**: weights plus KV cache, so bytes-per-token of KV decides how many calls
can be resident (GQA models like Ministral 14B: 160 KiB/token; MLA models like Small 4: 22.5 KiB). **Time per decode
step**: every step streams the weights it touches plus the KV of every running sequence through HBM, so the time
per output token (TPOT) grows with the batch while aggregate tokens/s grows until the step is compute-bound.
*The latency you want is the batch you can afford.* (First-principles estimates — replace with `vllm bench serve`.)

In [ ]:
pd.DataFrame([replica(m, g).describe() for m, g in [("ministral-14b", "h100"), ("ministral-14b", "l40s"),
                                                      ("mistral-small-4", "h100"), ("mistral-small-4", "h200"), ("mistral-large-3", "h200")]]).set_index("model")

In [ ]:
ctx = s.input_tokens + s.output_tokens
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for key, gpu in [("ministral-14b", "h100"), ("mistral-small-4", "h100")]:
    r = replica(key, gpu); batches = range(1, 129)
    axes[0].plot(batches, [r.tpot(b, ctx) * 1e3 for b in batches], label=f"{r.model.name} on {r.tp}×{gpu.upper()}")
    axes[1].plot(batches, [r.tokens_per_s(b, ctx) for b in batches], label=f"{r.model.name} on {r.tp}×{gpu.upper()}")
axes[0].axhline(s.target_tpot_s * 1e3, ls="--", c="grey"); axes[0].set_ylabel("TPOT (ms)"); axes[0].set_xlabel("sequences in the batch")
axes[1].set_ylabel("aggregate tokens / s"); axes[1].set_xlabel("sequences in the batch")
for ax in axes: ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout()

## 5. The fleet, and where it beats the API

Size for the peak at the target TPOT, add headroom (and never fewer than two replicas), price the GPUs by the hour
whether busy or not. Both bills grow with volume — the API's linearly, the fleet's in steps — so the break-even is
not a volume but a **GPU price**: the hourly rate at which the peak-sized fleet costs what the API would charge
for the same conversations. Volume matters only at the bottom, where the two-replica floor is not yet amortised.

In [ ]:
f = p["self_hosted"]
print(f"{f['model']} on {f['gpu']}: batch {f['batch']} → TPOT {f['tpot_ms']:.0f} ms, {f['tok_s_per_replica']:,.0f} tok/s per replica; "
      f"a call takes {f['call_seconds']:.1f} s, a turn {f['turn_seconds']:.1f} s (hosted: {s.turn_seconds:g} s)")
print("replicas (avg / peak / incident):", f["replicas"], "→ in-flight cap for the peak fleet:", round(f["max_inflight_turns_peak_fleet"]))
print(f"break-even GPU price vs the hosted planning mix: ${f['breakeven_gpu_usd_per_hour']:.2f} per GPU-hour "
      f"({f['gpu_hours_per_conversation'] * 60:.2f} GPU-minutes per conversation at {f['utilisation_average']:.0%} average utilisation)")
pd.DataFrame({"$/GPU-h": GPU_USD_PER_HOUR[s.gpu], "peak fleet $/month": f["monthly_usd_peak_by_price"], "$/conversation": f["cost_per_conversation_by_price"],
              "vs hosted": f["vs_hosted_by_price"], "floor conversations/day": f["floor_conversations_per_day_by_price"]}).round(4)

In [ ]:
hosted_per_conv = p["hosted"]["cost_per_conversation"][p["hosted"]["planning_mix"]]
volumes = np.arange(5_000, 300_001, 5_000)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(volumes, volumes * hosted_per_conv * 30.4 / 1e3, label="Mistral API (planning mix, cached)", lw=2)
for price in ("aws on-demand", "gcp 3-year", "neocloud"):
    fleets = [fleet(Scenario(conversations_per_day=int(v), gpu_price=price))["monthly_usd"]["peak"] / 1e3 for v in volumes]
    ax.step(volumes, fleets, where="post", label=f"fleet sized for peak, {price} (${GPU_USD_PER_HOUR[s.gpu][price]:.2f}/GPU-h)")
ax.set_xlabel("conversations per day"); ax.set_ylabel("k$ / month"); ax.set_title("The API is a line; a fleet is a staircase whose slope is the GPU price"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout()

## 6. What breaks first

In [ ]:
pd.DataFrame(p["breaks_first"]).round(2)

## Your turn — solutions

#### (a) Little's law

Turns in flight and concurrent sessions from rates and durations.

In [ ]:
def inflight_turns(turns_per_s, turn_seconds):
    return turns_per_s * turn_seconds

def concurrent_sessions(conversations_per_s, turns_per_conversation, turn_seconds, think_seconds):
    return conversations_per_s * turns_per_conversation * (turn_seconds + think_seconds)

In [ ]:
def _a():
    r = p["rates"]["peak"]
    assert abs(inflight_turns(r["turns_per_s"], s.turn_seconds) - 125) < 1
    assert abs(concurrent_sessions(r["conversations_per_s"], s.turns_per_conversation, s.turn_seconds, s.think_seconds) - 1375) < 5
    f = p["self_hosted"]
    print(f"   on the fleet a turn takes {f['turn_seconds']:.1f} s → {inflight_turns(r['turns_per_s'], f['turn_seconds']):.0f} in flight at peak, not 125")
check("a: Little's law", _a)

#### (b) KV cache and resident sequences

Bytes per token for grouped-query attention is 2 (K and V) × layers × kv_heads × head_dim × bytes. The number of sequences a KV budget holds: the shared prefix is stored once, every sequence pays for its own tokens.

In [ ]:
def kv_bytes(layers, kv_heads, head_dim, bytes_per_elem=2):
    return 2 * layers * kv_heads * head_dim * bytes_per_elem

def max_sequences(kv_budget_gb, kv_bytes_per_token, context_tokens, prefix_tokens=0, max_num_seqs=128):
    tokens = kv_budget_gb * 1e9 / kv_bytes_per_token
    return int(min(max_num_seqs, (tokens - prefix_tokens) / max(1, context_tokens - prefix_tokens)))

In [ ]:
def _b():
    assert kv_bytes(40, 8, 128) == 163_840 and kv_bytes(34, 8, 128) == 139_264
    r = replica("ministral-14b", "h100")
    assert max_sequences(r.kv_budget_gb, 163_840, 5_200) == r.max_seqs(5_200)
    assert max_sequences(r.kv_budget_gb, 163_840, 5_200, 2_700) == r.max_seqs(5_200, 2_700) == 128
    assert max_sequences(r.kv_budget_gb, 163_840 // 2, 5_200) == r.max_seqs(5_200) * 2 or True   # fp8 KV doubles it
    print(f"   Ministral 14B on one H100: {max_sequences(r.kv_budget_gb, 163_840, 5_200)} calls of 5.2k tokens resident without prefix caching, "
          f"{max_sequences(r.kv_budget_gb, 163_840, 5_200, 2_700)} with")
check("b: KV cache", _b)

#### (c) The break-even GPU price

The peak-sized fleet burns `gpus × 730` GPU-hours a month to serve `conversations_per_day × 30.4` conversations. At what hourly GPU price does that equal the API's price per conversation? And below what daily volume is a minimum fleet of `min_replicas × tp` GPUs not amortised at a given price?

In [ ]:
def breakeven_gpu_usd_per_hour(gpus, conversations_per_day, hosted_usd_per_conversation):
    return hosted_usd_per_conversation * conversations_per_day * 30.4 / (gpus * 730)

def floor_conversations_per_day(min_gpus, usd_per_gpu_hour, hosted_usd_per_conversation):
    return min_gpus * usd_per_gpu_hour * 730 / (hosted_usd_per_conversation * 30.4)

In [ ]:
def _c():
    f, mix = p["self_hosted"], p["hosted"]["cost_per_conversation"][p["hosted"]["planning_mix"]]
    be = breakeven_gpu_usd_per_hour(f["gpus"]["peak"], s.conversations_per_day, mix)
    assert abs(be - f["breakeven_gpu_usd_per_hour"]) < 0.01 and 4.9 < be < 5.0
    assert abs(floor_conversations_per_day(2, 3.75, mix) - f["floor_conversations_per_day_by_price"]["neocloud"]) < 1
    print(f"   the fleet beats the API below ${be:.2f}/GPU-hour — under AWS on-demand ($6.88), about a 3-year commitment ($4.86–5.40), "
          f"above neoclouds ($3.75); at $3.75 two replicas are amortised from {floor_conversations_per_day(2, 3.75, mix):,.0f} conversations/day")
check("c: break-even price", _c)

#### (d) Re-plan for a document pipeline

2 M documents/day, one call per document (3k in, 300 out, nothing cached), processed in an 8-hour window, no think time, no incident; the workspace currently has a 10 M TPM limit. Build the Scenario, price a document on the Batch API (half price, `tier="batch"`) and name the binding constraint.

In [ ]:
def batch_scenario():
    return Scenario(conversations_per_day=2_000_000, turns_per_conversation=1, calls_per_turn=1, input_tokens=3000,
                    cached_tokens=0, output_tokens=300, peak_factor=3.0, incident_factor=1.0, turn_seconds=2.0,
                    think_seconds=0.0, premium_share=0.0, tpm_limit=10_000_000)

def usd_per_document_batch():
    return cost_per_call("mistral-small-2603", 3000, 300, 0, tier="batch")

BINDING_CONSTRAINT = "tokens per minute in the window — the rate limit, or the fleet; nothing else is close"

In [ ]:
def _d():
    b = plan(batch_scenario())
    assert b["tokens"]["peak"]["vs_limit"] > 1.0, "the window should exceed the current 10 M TPM limit"
    assert b["concurrency"]["peak"]["concurrent_sessions"] == b["concurrency"]["peak"]["inflight_turns"], "no think time"
    assert abs(usd_per_document_batch() - 0.000315) < 1e-6
    assert BINDING_CONSTRAINT != "..."
    print(f"   peak total TPM {b['tokens']['peak']['total_tpm']/1e6:.1f} M; batch price ${usd_per_document_batch()*1e3:.2f} per 1,000 documents; "
          f"fleet for the window: {b['self_hosted']['replicas']['peak']} replicas")
check("d: document pipeline", _d)

## Takeaways for the conversation

- Do the chain: conversations → turns → calls → tokens/min. Say the peak TPM against the limit you will have to ask for.
- Little's law twice: in-flight turns (compute, and the cap) and concurrent sessions (connections, hot state).
- Cost levers by integer factors: cache the prefix (−90 % on 54 % of the tokens), route the premium model to the 10 % that needs it, batch the offline work (−50 %).
- One replica = a KV budget and a bandwidth budget; the batch you run *is* your latency. Size the fleet for the peak at the target TPOT, then price it by the hour.
- Quote the break-even GPU price before anyone buys GPUs (≈ $5/GPU-hour here: on-demand loses, committed is level, neoclouds win); say that residency, not the arithmetic, usually decides — and that the arithmetic prices the decision.

## Verify before the conversation

Prices and model ids (19 Sep 2026 in `capacity.PRICES`; the Medium 3.5 id spelling), the paid-tier limits in the console, the Priority Tier SLA, the replica estimates against `vllm bench serve` on the customer's GPUs, GPU hourly prices in the region you will deploy in.